# H4: CRNN: достаточно ли CNN + BiLSTM + CTC?

**Проверяемая гипотеза:** Предобученный CRNN уступает PP-OCRv6 small не больше чем на 2 п.п.

Четыре свёрточных блока → последовательность столбцов → двунаправленная LSTM → linear → CTC. LSTM проходит последовательность в двух направлениях; выходные символы не генерируются токен за токеном с языковым декодером. Проверяется готовый Graf-J checkpoint, без дообучения на нашем test.

[Теория и исследования](../research/THEORY_AND_PAPERS.md). [Зафиксированный протокол](../research/PROTOCOL.md). Пороговые значения — наши инженерные критерии, не требования заказчика.

In [1]:
from pathlib import Path
import sys, json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'captcha_lab').is_dir())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import pandas as pd
from captcha_lab.evaluation import (manifest,run_local,metrics,scored,
                                      compare,all_summaries,hypothesis_results,RESULTS)
audit=json.loads((ROOT/'datasets/captcha-images/audit.json').read_text())
assert audit['holdout_images']==1984 and audit['shared_images']==200
display(pd.DataFrame([{'split':k,'n':v} for k,v in audit['source_split_counts'].items()]))
print('Holdout manifest SHA-256:',audit['holdout_manifest_sha256'])
print('Shared manifest SHA-256:',audit['shared_manifest_sha256'])
print('No prediction caches: inference cell below executes the actual model.')

,split,n
0,train,6000
1,validation,2000
2,test,2000


Holdout manifest SHA-256: 33fe043e8dec7b0e238c6bc81bb63c601699540b4b6c2025ad9480e21164f42b
Shared manifest SHA-256: 675bc9b4d93369e68817412fa08a72c6e21de23c820e00e42bc30d06c7390384
No prediction caches: inference cell below executes the actual model.


## Размеченная выборка

Публичный полный test содержит 2000 изображений; 16 просмотренных в предыдущем пилоте исключены. Метки и архивы скачаны полностью, не из preview. Функция распознавания получает только пиксели; метки присоединяются после предсказания. Исходные семь целевых меток выделены отдельно, два спорных примера исключены.

In [2]:
labels=manifest('shared_200' if 'crnn-base'=='gemini' else 'holdout')
display(labels[['image_id','label','image_path']].head(12))
print('Evaluation images:',len(labels))

,image_id,label,image_path
0,test_000c1b6d4e5f11fb75d55a7c094b901e,161660,data/benchmark/images/test/161660.000c1b6d4e5f...
1,test_00105a4c053053509955d95368312987,496670,data/benchmark/images/test/496670.00105a4c0530...
2,test_001a3b40ef793b5984df8969f145d7bb,744782,data/benchmark/images/test/744782.001a3b40ef79...
3,test_0021e642491cf5bbc13cfc59fa46084a,017232,data/benchmark/images/test/017232.0021e642491c...
4,test_0047646a0e3b67ebc360e43ba88d1596,853091,data/benchmark/images/test/853091.0047646a0e3b...
5,test_0064b2188592a16991d5394b3ec5bf01,959686,data/benchmark/images/test/959686.0064b2188592...
6,test_00a5d81ab5060a0d45608169ce20bda9,159645,data/benchmark/images/test/159645.00a5d81ab506...
7,test_00e02c7c9da2cc0d3cdf5e3d9183a9f5,952376,data/benchmark/images/test/952376.00e02c7c9da2...
8,test_0123f4327a176c460aeefecd1d69bc11,374639,data/benchmark/images/test/374639.0123f4327a17...
9,test_0127972a796c1f306fdb5f919a7649fa,517493,data/benchmark/images/test/517493.0127972a796c...


Evaluation images: 1984


## Реальный прогон

Ячейка заново загружает модель и выполняет inference на каждой картинке. Ошибки сохраняются, готовые ответы не подставляются. CPU, batch=1, отдельный прогрев на validation.

In [3]:
frames=run_local('crnn-base')

C:\Users\cvind\Documents\Codex\2026-10-21\files-mentioned-by-the-user-c\work\Captcha-MAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


crnn-base holdout 200 / 1984


crnn-base holdout 400 / 1984


crnn-base holdout 600 / 1984


crnn-base holdout 800 / 1984


crnn-base holdout 1000 / 1984


crnn-base holdout 1200 / 1984


crnn-base holdout 1400 / 1984


crnn-base holdout 1600 / 1984


crnn-base holdout 1800 / 1984


In [4]:
rows=[]
for dataset,predictions in frames.items():
    labels=manifest(dataset)
    rows.append(dict(dataset=dataset,**metrics(predictions,labels)))
    joined=scored(predictions,labels)
    print(dataset,'— ошибки по изображениям:')
    display(joined.loc[~joined.correct,['image_id','label','prediction','error','error_detail']].head(20))
display(pd.DataFrame(rows))

holdout — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
0,test_000c1b6d4e5f11fb75d55a7c094b901e,161660,I61660,,
2,test_001a3b40ef793b5984df8969f145d7bb,744782,74482,,
3,test_0021e642491cf5bbc13cfc59fa46084a,017232,07232,,
5,test_0064b2188592a16991d5394b3ec5bf01,959686,99686,,
6,test_00a5d81ab5060a0d45608169ce20bda9,159645,750645,,
7,test_00e02c7c9da2cc0d3cdf5e3d9183a9f5,952376,95236,,
9,test_0127972a796c1f306fdb5f919a7649fa,517493,57493,,
11,test_0159b035fbb7d58b185c92b964eec949,130021,730027,,
13,test_01e828220fa7d238b62f51fbdc47409d,170370,j0370,,
15,test_0214693960836c1e7500ee8ed61de40a,906192,9OO2,,


target — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
0,33ef91d464c03eb9,894412,8344z,,
1,41ada5001f1e1e88,432077,482ui,,
2,5cb6a434364bfe27,558796,55816,,
3,7619f1f1b372df6d,181263,131263,,
4,8731a1e555abd295,926325,g2635,,
5,d4c0652824fb2dfd,639125,6z,,
6,d4d1163a8ce99293,642044,64324,,


,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
0,holdout,1984,665,0.335181,0.314746,0.356254,0.201865,0,0,16.3400,19.86527,16.3400,19.86527
1,target,7,0,0.000000,0.000000,0.354330,0.476190,0,0,22.4267,23.51206,22.4267,23.51206


## Вывод по гипотезе

Exact match и CER вычисляются по скачанным меткам. Вердикт получается из заранее заданного критерия и интервала разности; отсутствие статистической определённости явно показывается. Правильность на публичном наборе не переносится автоматически на Instagram.

In [5]:
summary=all_summaries()
display(summary[summary.model.eq('crnn-base')])
hypotheses,paired=hypothesis_results()
display(hypotheses[hypotheses.hypothesis.eq('H4')])
display(paired[paired.hypothesis.eq('H4')] if not paired.empty else paired)

,model,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
9,crnn-base,holdout,1984,665,0.335181,0.314746,0.356254,0.201865,0,0,16.3400,19.865270,16.3400,19.865270
10,crnn-base,shared_200,200,76,0.380000,0.315590,0.448933,0.180833,0,0,16.2025,19.286505,16.2025,19.286505
11,crnn-base,target,7,0,0.000000,0.000000,0.354330,0.476190,0,0,22.4267,23.512060,22.4267,23.512060


,hypothesis,criterion,verdict,estimate,ci_low,ci_high
3,H4,delta crnn-base-ppocr-small>=-2.00%,rejected,-0.610383,-0.638402,-0.57928


,hypothesis,model_a,model_b,n,wins,losses,delta,delta_ci95_low,delta_ci95_high,mcnemar_exact_p,holm_p
2,H4,crnn-base,ppocr-small,1984,20,1231,-0.610383,-0.638402,-0.57928,0.0,0.0
